# LegalQA Main 1/3 — QLoRA train và chạy tiếp trên Kaggle

Chọn **GPU T4 x2**. Add Input dataset BTC và Version 3 chứa `index/` + `models/`.
Notebook tự tìm dưới `/kaggle/input`, không phụ thuộc tên/mức lồng thư mục của Kaggle.

**Chạy tiếp:** Save Version có output; lần sau Add Input **toàn bộ output notebook lần trước**, giữ Input Version 3 và để `INPUT_MODE='auto'`. Tự khóa commit, kiểm tra checksum/fingerprint, copy tiến độ sang `/kaggle/working`, dùng lại retrieval hoàn tất (hoặc journal dở dang), chọn checkpoint hợp lệ có global_step cao nhất và khôi phục optimizer/RNG cả hai rank. Không cần dataset gốc nếu output đã có split đầy đủ.

`auto` ưu tiên resume output Stage 1; chỉ có diagnostics ZIP thì import retrieval cho lượt train mới. `resume` bắt buộc có output; `retrieval` tự tìm cache cũ/ZIP, dùng code mới và bắt đầu optimizer mới; `fresh` bỏ qua output cũ. Nhiều nguồn phù hợp: dừng và in đường dẫn để bạn chọn ROOT, không đoán phiên. Cache 768 QA không được trộn vào lượt 5.600 QA.

**RAM/GPU:** code mới đọc cache từng bản ghi, nén token ngay khi tạo, nạp hai model lần lượt; QLoRA NF4, fused loss, checkpointing, batch 1/GPU × accumulation 4 × 2 GPU = 8. Giữ toàn bộ target và giới hạn 8192 token. Dừng/lưu khi RAM thấp; supervisor dừng cả nhóm worker dưới ngưỡng khẩn cấp. Không thể cam kết không OOM trước khi đo trên dữ liệu/GPU thật. BM25 là tác vụ CPU; DDP proof xác nhận optimizer chạy trên cả hai GPU.

**Commit cũ:** resume giữ code của checkpoint để bảo toàn tiến độ; không tự ghép bản sửa loss/RAM vào optimizer cũ. Muốn áp dụng code mới cho output khác commit, dùng `INPUT_MODE='retrieval'` (train lại, tái sử dụng BM25). Push bản sửa lên repo được clone trước khi bắt đầu lượt mới.

Tối đa 9 giờ từ cell đầu + 10 phút export. `paused` là snapshot hợp lệ; diagnostics ZIP chỉ phục vụ kiểm tra, không có trọng số. Chỉ sang Stage 2 khi `STATUS: complete`. Chi tiết và smoke test: `docs/main01_resume.md`.


In [ ]:
import json, os, signal, subprocess, sys, time
from pathlib import Path

# Count setup/install time too. Do not reset this timestamp in later cells.
SESSION_STARTED = time.monotonic()
if not Path('/kaggle').is_dir():
    raise RuntimeError('Notebook chỉ chạy trên Kaggle.')
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
CODE = WORK / 'legalqa_stage_code'

# 9h includes setup and compute; export gets up to 10 additional minutes.
# The remaining margin is reserved for Kaggle output collection and runtime variation.
WORK_HOURS = 9.0
EXPORT_SECONDS = 600
MIN_FREE_RAM_MB = 3072  # Cooperative save threshold; hard stop below 1536 MiB.
VERSION3_ROOT = None     # Auto-find index + models anywhere under /kaggle/input.
DATASET_ROOT = None      # Auto-find train.json + public-official.json; optional on resume.

# None = discover exactly one matching input; set a full ROOT if multiple versions exist.
INPUT_MODE = 'auto'     # auto / resume / retrieval / fresh; auto prioritizes full output.
PREVIOUS_OUTPUT = None   # Cumulative output of THIS stage from an earlier session.
UPSTREAM_OUTPUT = None   # Stage 1 for notebook 02; Stage 2 for notebook 03.
LEGACY_INPUT_ROOT = None # Notebook 01 only: old legalqa_quality_v8_full with completed QLoRA.
RETRIEVAL_INPUT = None   # Stage 1: old output folder or diagnostics ZIP; reuse retrieval only.
REPO_REVISION = None     # First run: main. Continuations: automatically pin upstream commit.

STAGE = 1
MODE = 'auto'
MAX_NEW_QUESTIONS = 200

# Fail before retrieval if the requested Kaggle accelerator is unavailable.
gpu_names = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, timeout=30
).strip().splitlines()
if len(gpu_names) != 2 or not all('T4' in name for name in gpu_names):
    raise RuntimeError(f'Chọn accelerator GPU T4 x2 trước khi chạy Stage 1; hiện có: {gpu_names}')
print('Training GPUs:', gpu_names)


## Tự tìm input, khóa commit và kiểm soát phiên

Giữ một output của lượt cần resume trong Input. `PREVIOUS_OUTPUT` nhận cả ROOT hoặc thư mục mount bao ngoài.


In [ ]:
if not 0 < WORK_HOURS <= 9:
    raise ValueError('WORK_HOURS phải trong (0, 9]; giữ thời gian dự phòng trước 12h.')
WORK_END = SESSION_STARTED + WORK_HOURS * 3600

def find_root(value, marker_name, required_files, label, required=True):
    if value is not None:
        explicit = Path(value)
        if not explicit.exists():
            raise FileNotFoundError(explicit)
        search = explicit.parent if explicit.is_file() else explicit
    else:
        search = INPUT
    candidates = sorted({p.parent for p in search.rglob(marker_name)
                         if all((p.parent/name).is_file() for name in required_files)})
    if len(candidates) > 1:
        raise RuntimeError(f'Nhiều {label}: {candidates}. Chỉ định ROOT trong cell cấu hình.')
    if candidates:
        return candidates[0]
    if required or value is not None:
        raise FileNotFoundError(f'Không tìm thấy {label} trong {search}; cần {required_files}')
    return None

def resolve_output(value, stage, required=False):
    return find_root(value, f'stage{stage}_manifest.json',
                     [f'stage{stage}_manifest.json'], f'output Stage {stage}', required)

def resolve_retrieval(value=None):
    from zipfile import ZipFile, BadZipFile
    search = Path(value) if value is not None else INPUT
    if not search.exists():
        raise FileNotFoundError(search)
    if search.is_file() and search.suffix.lower() != '.zip':
        search = search.parent
    sources = []
    manifests = [search/'stage1_manifest.json'] if search.is_dir() and (search/'stage1_manifest.json').is_file() else list(search.rglob('stage1_manifest.json')) if search.is_dir() else []
    for manifest in manifests:
        info = json.loads(manifest.read_text(encoding='utf-8'))
        name = 'train.sft.lexical.retrieval.json'
        if name in info.get('files', {}) and (manifest.parent/name).is_file():
            sources.append(manifest.parent)
    # Do not count a diagnostics ZIP twice when the full folder is attached.
    archives = [search] if search.is_file() else sorted(search.rglob('*.zip'))
    for archive in archives:
        if any(root == archive.parent or root in archive.parents for root in sources):
            continue
        try:
            with ZipFile(archive) as z:
                names = z.namelist()
                if 'stage1_manifest.json' in names and 'train.sft.lexical.retrieval.json' in names:
                    sources.append(archive)
        except BadZipFile:
            continue
    if len(sources) != 1:
        raise RuntimeError(f'Cần đúng một nguồn retrieval hoàn tất, tìm thấy: {sources}. Chỉ định RETRIEVAL_INPUT.')
    return sources[0]

if INPUT_MODE not in {'auto', 'resume', 'retrieval', 'fresh'}:
    raise ValueError('INPUT_MODE phải là auto/resume/retrieval/fresh.')
if UPSTREAM_OUTPUT is not None:
    raise ValueError('Stage 1 không nhận UPSTREAM_OUTPUT.')
if INPUT_MODE == 'fresh':
    if any(v is not None for v in (PREVIOUS_OUTPUT, RETRIEVAL_INPUT, LEGACY_INPUT_ROOT)):
        raise ValueError('fresh không được trộn với previous/retrieval/legacy.')
elif RETRIEVAL_INPUT is not None or INPUT_MODE == 'retrieval':
    if INPUT_MODE == 'resume' or PREVIOUS_OUTPUT is not None or LEGACY_INPUT_ROOT is not None:
        raise ValueError('Retrieval-only import không resume checkpoint; bỏ previous/legacy.')
    RETRIEVAL_INPUT = resolve_retrieval(RETRIEVAL_INPUT)
elif LEGACY_INPUT_ROOT is None:
    PREVIOUS_OUTPUT = resolve_output(PREVIOUS_OUTPUT, STAGE, required=INPUT_MODE == 'resume')
    if PREVIOUS_OUTPUT is None and INPUT_MODE == 'auto':
        from zipfile import ZipFile, BadZipFile
        diagnostics = []
        for path in INPUT.rglob('*.zip'):
            try:
                with ZipFile(path) as z:
                    if 'stage1_manifest.json' in z.namelist():
                        diagnostics.append(path)
            except BadZipFile:
                continue
        if diagnostics:
            RETRIEVAL_INPUT = resolve_retrieval()
if LEGACY_INPUT_ROOT is not None:
    LEGACY_INPUT_ROOT = Path(LEGACY_INPUT_ROOT)
    if PREVIOUS_OUTPUT is not None or INPUT_MODE == 'resume':
        raise ValueError('Không trộn legacy import với resume.')

# The old dataset mount name is irrelevant. Detect by the files actually used.
VERSION3_ROOT = find_root(VERSION3_ROOT, 'index_manifest.json',
    ['index_manifest.json', 'corpus.sqlite'], 'Version 3 index').parent
for name in ['models/models.lock.json'] + [f'models/{role}/config.json' for role in ('embedding', 'reranker', 'generator')]:
    if not (VERSION3_ROOT/name).is_file():
        raise FileNotFoundError(VERSION3_ROOT/name)
for role in ('embedding', 'reranker', 'generator'):
    if not any((VERSION3_ROOT/'models'/role).glob('*.safetensors')):
        raise FileNotFoundError(f'Thiếu trọng số {role} trong {VERSION3_ROOT}/models')
has_split = PREVIOUS_OUTPUT is not None and (PREVIOUS_OUTPUT/'data/split_manifest.json').is_file()
if DATASET_ROOT is not None or not has_split:
    DATASET_ROOT = find_root(DATASET_ROOT, 'train.json', ['train.json', 'public-official.json'], 'dataset BTC')
print('Resolved dataset:', DATASET_ROOT, '| Version 3:', VERSION3_ROOT)
print('Input mode:', INPUT_MODE, '| Retrieval-only:', RETRIEVAL_INPUT)

pins = []
for source, number in [(PREVIOUS_OUTPUT, STAGE), (UPSTREAM_OUTPUT, STAGE - 1)]:
    if source is not None:
        info = json.loads((source / f'stage{number}_manifest.json').read_text(encoding='utf-8'))
        if info.get('schema') != 2:
            raise ValueError('Input dùng schema cũ. Chọn đúng output mới hoặc legacy import ở Stage 1.')
        pins.append(info['code_commit'])
if len(set(pins)) > 1:
    raise ValueError('Upstream và previous output khác code commit.')
PIN = pins[0] if pins else (REPO_REVISION or 'main')
if pins and REPO_REVISION and REPO_REVISION != PIN:
    raise ValueError('Không đổi commit khi resume. Bắt đầu một experiment mới nếu cần đổi code.')

def available_ram_mb(proc=Path('/proc/meminfo'), cgroup=Path('/sys/fs/cgroup')):
    values = {}
    if proc.is_file():
        values = {line.split(':')[0]: int(line.split()[1])*1024
                  for line in proc.read_text().splitlines() if ':' in line}
    available = [values['MemAvailable']] if 'MemAvailable' in values else []
    for limit_file, usage_file in (('memory.max', 'memory.current'),
                                  ('memory/memory.limit_in_bytes', 'memory/memory.usage_in_bytes')):
        try:
            limit = (cgroup/limit_file).read_text().strip()
            if limit != 'max':
                available.append(max(0, int(limit)-int((cgroup/usage_file).read_text())))
        except FileNotFoundError:
            pass
    return min(available)/1024**2 if available else None


class BudgetPause(Exception):
    pass

def bounded_process(command, *, seconds=None, cwd=None, env=None):
    remaining = WORK_END - time.monotonic()
    if remaining <= 0:
        raise BudgetPause('Đã hết ngân sách phiên.')
    limit = remaining if seconds is None else min(remaining, seconds)
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), cwd=cwd, env=env, start_new_session=True)
    stop_at = time.monotonic() + limit
    try:
        while True:
            free = available_ram_mb()
            if free is not None and free < MIN_FREE_RAM_MB / 2:
                raise MemoryError(f'RAM guard: còn {free:.0f} MiB; dừng worker để giữ snapshot.')
            left = stop_at - time.monotonic()
            if left <= 0:
                raise subprocess.TimeoutExpired(command, limit)
            try:
                rc = process.wait(timeout=min(left, 2))
                break
            except subprocess.TimeoutExpired:
                continue
    except (subprocess.TimeoutExpired, KeyboardInterrupt, MemoryError) as error:
        # Worker and every legalqa subprocess share this process group.
        # Stop all of them before hashing/exporting artifacts.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            pass
        # The group leader may exit while a GPU child ignores SIGTERM.
        # Always kill remaining group members before exporting.
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=30)
        if isinstance(error, KeyboardInterrupt):
            raise
        raise BudgetPause(str(error) if isinstance(error, MemoryError) else 'Đã dừng worker theo ngân sách; tiến độ đã ghi sẽ được export.') from error
    if rc:
        raise subprocess.CalledProcessError(rc, command)

if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} không phải repo. Dùng phiên Kaggle mới.')
    remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True, timeout=30).strip()
    if remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError('Repo origin không khớp.')
    dirty = subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain'], text=True, timeout=30).strip()
    if dirty:
        raise RuntimeError('Code trong session có sửa đổi; không tự ghi đè. Dùng phiên mới.')
else:
    bounded_process(['git', 'clone', '--no-checkout', '--depth', '1', REPO_URL, CODE], seconds=300)
bounded_process(['git', '-C', CODE, 'fetch', '--depth', '1', 'origin', PIN], seconds=300)
bounded_process(['git', '-C', CODE, 'checkout', '--detach', 'FETCH_HEAD'], seconds=60)
commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True, timeout=30).strip()
if pins and commit != PIN:
    raise RuntimeError('Checkout không đúng commit đã khóa.')
if not (CODE / 'legalqa' / 'stages.py').is_file():
    raise RuntimeError('Commit chưa có stages.py. Push các thay đổi mới trước khi chạy.')
print('Pinned commit:', commit)
print('Previous:', PREVIOUS_OUTPUT, '| Upstream:', UPSTREAM_OUTPUT)

# Do not silently run an old one-GPU training config when continuing an output.
runtime_config = json.loads((CODE/'config.json').read_text(encoding='utf-8'))
if runtime_config['training'].get('world_size', 1) != 2:
    raise RuntimeError('Checkpoint khóa vào code train 1 GPU. Không đổi sang DDP giữa lượt; dùng INPUT_MODE="retrieval" cho lượt mới 2 GPU.')
if PREVIOUS_OUTPUT is not None:
    print('Exact resume: giữ commit trong manifest; các sửa mới chỉ áp dụng nếu đã có trong commit đó.')


## Cài môi trường và kiểm tra


In [ ]:
bounded_process([sys.executable, '-m', 'pip', 'install', '-q', '-r', CODE / 'requirements.txt'], seconds=1200)
if STAGE == 2:
    bounded_process([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], seconds=300)
    bounded_process([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, seconds=300)
bounded_process([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, seconds=300)


## Chạy trong ngân sách và export cả tiến độ dở dang

Mọi xử lý/kiểm tra artifact dùng legalqa/stages.py chung cho ba notebook. Diagnostics chứa dữ liệu dev, reference, retrieval, prediction, audit, metrics và trạng thái train đã có; không chứa trọng số.


In [ ]:
RUN_ROOT = WORK / f'legalqa_main_stage{STAGE}_v8'
OPTIONS = WORK / f'legalqa_stage{STAGE}_options.json'
options = {
    'stage': STAGE, 'root': str(RUN_ROOT), 'version3': str(VERSION3_ROOT),
    'dataset': str(DATASET_ROOT) if DATASET_ROOT is not None else None, 'mode': MODE, 'max_new_questions': MAX_NEW_QUESTIONS,
    'previous': str(PREVIOUS_OUTPUT) if PREVIOUS_OUTPUT is not None else None,
    'upstream': str(UPSTREAM_OUTPUT) if UPSTREAM_OUTPUT is not None else None,
    'legacy': str(LEGACY_INPUT_ROOT) if LEGACY_INPUT_ROOT is not None else None,
    'retrieval_input': str(RETRIEVAL_INPUT) if RETRIEVAL_INPUT is not None else None,
}
OPTIONS.write_text(json.dumps(options, ensure_ascii=False, indent=2), encoding='utf-8')
# Cooperative pause 10 minutes before hard worker stop, for saving Trainer state.
remaining = max(0, WORK_END - time.monotonic())
worker_env = {**os.environ, 'LEGALQA_DEADLINE': str(time.time() + max(0, remaining - 600)),
              'PYTHONUNBUFFERED': '1', 'LEGALQA_MIN_FREE_RAM_MB': str(MIN_FREE_RAM_MB),
              'TOKENIZERS_PARALLELISM': 'false'}
outcome, failure = 'ok', None
try:
    bounded_process([sys.executable, '-m', 'legalqa.stages', 'run', '--options', OPTIONS],
                    cwd=CODE, env=worker_env)
except BudgetPause as error:
    outcome = 'paused'
    print(str(error), flush=True)
except Exception as error:
    outcome, failure = 'failed', error
finally:
    if (RUN_ROOT / 'session.json').is_file():
        # Export runs only after the entire compute process group has stopped.
        # It is bounded separately, without restarting the 9h compute budget.
        original_end = WORK_END
        WORK_END = min(SESSION_STARTED + 10 * 3600, time.monotonic() + EXPORT_SECONDS)
        try:
            bounded_process([sys.executable, '-m', 'legalqa.stages', 'finalize',
                             '--options', OPTIONS, '--outcome', outcome], cwd=CODE)
        finally:
            WORK_END = original_end
if failure is not None:
    raise failure
manifest_path = RUN_ROOT / f'stage{STAGE}_manifest.json'
if not manifest_path.is_file():
    raise RuntimeError('Chưa tạo được snapshot; xem lỗi setup ở trên.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('STATUS:', manifest['status'])
print('PROGRESS:', manifest['progress'])
print('OUTPUT:', RUN_ROOT)
if manifest['status'] == 'complete':
    print('Stage hoàn tất. Có thể dùng output cho stage tiếp theo.')
else:
    print('Phiên kết thúc có chủ đích. Save output, Add Input vào CÙNG notebook, rồi chạy tiếp.')
proof_path = RUN_ROOT / 'sft/distributed_training.json'
if proof_path.is_file():
    print('DDP proof:', json.loads(proof_path.read_text(encoding='utf-8')))
print('Lần sau: giữ Input Version 3 (index/models), Add Input toàn bộ OUTPUT ở trên; INPUT_MODE=auto.')
print('Diagnostics ZIP không có trọng số/optimizer, không thay thế output đầy đủ để resume.')
